# 12 — Inference: Prediction Demo

End-to-end tour of the `inference` package for single and batch prediction. Uses `generate_patient` to synthesise cases on the fly — no CSV required.

LLM explanations use the built-in `StubClient` (deterministic, offline). To run against Gemini, set `BANDITS_LLM_PROVIDER=gemini` and `GEMINI_API_KEY`.

## 1. Setup

In [1]:
from pathlib import Path
import os, sys
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

import numpy as np
from src.data_generator import generate_patient, TREATMENTS
from inference import (
    InferenceEngine, InferenceConfig,
    PatientInput, ValidationError,
)


In [2]:
cfg = InferenceConfig.load(
    llm_enabled=True,
    llm_provider='stub',   # change to 'gemini' when a key is set
    n_confidence_draws=200,
    device='cpu',
)
engine = InferenceEngine.from_config(cfg)
engine.snapshot()

2026-04-16 09:08:46.819 | INFO     | inference._internal.feature_engineering:load:310 - FeaturePipeline loaded from models\feature_pipeline.joblib: 25 features, scale=True


2026-04-16 09:08:47.981 | INFO     | inference._internal.neural_bandit:__init__:158 - NeuralThompson initialized: input=25, hidden=[128, 64], device=cpu


2026-04-16 09:08:47.991 | INFO     | inference._internal.neural_bandit:load:1000 - Loaded NeuralThompson (with posterior) from models\neural_thompson.pt


2026-04-16 09:08:47.992 | INFO     | inference._internal.neural_bandit:enable_online_retraining:831 - Online retraining enabled: buffer=50000, retrain_every=2000, minibatch=1024


2026-04-16 09:08:47.993 | INFO     | inference._internal.explainability:__init__:557 - ExplainabilityExtractor initialized — confidence method: posterior sampling (200 draws), attribution=on, fairness=omitted


{'ready': True,
 'n_updates': 0,
 'model_path': 'models\\neural_thompson.pt',
 'pipeline_path': 'models\\feature_pipeline.joblib',
 'model_version': 'c9aaa56b2514',
 'pipeline_version': '96cb7eab5fe6',
 'feature_names': ['age',
  'bmi',
  'hba1c_baseline',
  'egfr',
  'diabetes_duration',
  'fasting_glucose',
  'c_peptide',
  'bp_systolic',
  'ldl',
  'hdl',
  'triglycerides',
  'alt',
  'cvd',
  'ckd',
  'nafld',
  'hypertension',
  'age_x_ckd',
  'bmi_x_nafld',
  'cvd_x_egfr',
  'cpeptide_x_hba1c',
  'duration_x_hba1c',
  'fg_x_bmi',
  'tg_hdl_ratio',
  'renal_risk',
  'severity_score'],
 'llm_enabled': True,
 'llm_provider': 'stub',
 'online_retraining': True,
 'drift_enabled': True,
 'n_confidence_draws': 200,
 'attribution_enabled': True}

## 2. Single prediction — raw output

In [3]:
rng = np.random.RandomState(42)
patient = generate_patient(rng)
patient['patient_id'] = 'DEMO-0001'
patient

{'age': 63,
 'bmi': 36.8,
 'hba1c_baseline': 7.49,
 'egfr': 92.6,
 'diabetes_duration': 2.6,
 'fasting_glucose': 171.2,
 'c_peptide': 1.93,
 'cvd': 1,
 'ckd': 0,
 'nafld': 0,
 'hypertension': 1,
 'bp_systolic': 130.6,
 'ldl': 95.9,
 'hdl': 28.0,
 'triglycerides': 193.4,
 'alt': 10.0,
 'patient_id': 'DEMO-0001'}

In [4]:
result = engine.predict(patient, explain=False)
print(f'Recommended: {result.recommended}')
print(f'Confidence:  {result.confidence_pct}% ({result.confidence_label})')
print(f'Safety:      {result.safety_status}')
print(f'Runner-up:   {result.runner_up} '
      f'(win rate {result.runner_up_win_rate:.2%})')
print()
print('Posterior means (pp HbA1c reduction):')
for t, v in result.posterior_means.items():
    print(f'  {t:<10s} {v:+.2f}')

C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
2026-04-16 09:08:48.070 | INFO     | inference._internal.explainability:extract_safety:676 - Safety check CLEAR for SGLT-2


2026-04-16 09:08:48.071 | INFO     | inference._internal.explainability:extract:738 - Extracted payload: final=SGLT-2, orig=SGLT-2, confidence=100% (HIGH), safety=CLEAR


Recommended: SGLT-2
Confidence:  100% (HIGH)
Safety:      CLEAR
Runner-up:   GLP-1 (win rate 0.00%)

Posterior means (pp HbA1c reduction):
  Metformin  +0.85
  GLP-1      +0.90
  SGLT-2     +1.53
  DPP-4      +0.61
  Insulin    +0.24


## 3. Prediction with LLM explanation (stub)

In [5]:
result = engine.predict(patient, explain=True)
print('RECOMMENDATION SUMMARY:')
print(result.explanation['recommendation_summary'])
print()
print('SAFETY ASSESSMENT:')
print(result.explanation['safety_assessment'])
print()
print('MONITORING NOTE:')
print(result.explanation['monitoring_note'])

C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
2026-04-16 09:08:48.136 | INFO     | inference._internal.explainability:extract_safety:676 - Safety check CLEAR for SGLT-2


2026-04-16 09:08:48.137 | INFO     | inference._internal.explainability:extract:738 - Extracted payload: final=SGLT-2, orig=SGLT-2, confidence=100% (HIGH), safety=CLEAR


2026-04-16 09:08:48.141 | INFO     | inference._internal.llm_explain:__init__:521 - LLMExplainer initialized: client=StubClient, model=gemini-2.0-flash, temp=0.3, max_retries=2


2026-04-16 09:08:48.142 | INFO     | inference._internal.llm_explain:explain:551 - Explanation generated (attempt 1): treatment=SGLT-2


RECOMMENDATION SUMMARY:
For this patient, the selected glucose-lowering therapy is SGLT-2. The predicted HbA1c reduction is modest and appropriate for the clinical profile described.

SAFETY ASSESSMENT:
Safety checks were applied per the structured findings provided; no additional concerns are introduced by this explanation.

MONITORING NOTE:
Monitor HbA1c at 3 months, renal function at 6 months, and review tolerability at the next visit.


## 4. Safety gate — eGFR < 30 forces an override

Metformin is contraindicated at eGFR < 30 (lactic-acidosis risk). Force a low eGFR and confirm the gate excludes Metformin from the final recommendation.

In [6]:
patient_ckd = dict(patient)
patient_ckd['egfr'] = 20.0     # severe CKD
patient_ckd['ckd'] = 1
result_ckd = engine.predict(patient_ckd)

print(f'Final recommendation: {result_ckd.recommended}')
print(f'Safety status:        {result_ckd.safety_status}')
if result_ckd.override is not None:
    print()
    print('-- OVERRIDE FIRED --')
    print(f"Original:  {result_ckd.override['original_treatment']}")
    print(f"Final:     {result_ckd.override['final_treatment']}")
    print(f"Reason:    {result_ckd.override['reason']}")
    print(f"Blocked:   {result_ckd.override['blocked_treatments']}")

assert result_ckd.recommended != 'Metformin', 'safety gate failed'
print()
print('assertion passed: Metformin is not the final recommendation')

C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


2026-04-16 09:08:48.202 | INFO     | inference._internal.explainability:extract_safety:674 - Safety warnings for DPP-4


2026-04-16 09:08:48.203 | INFO     | inference._internal.explainability:extract:738 - Extracted payload: final=DPP-4, orig=DPP-4, confidence=100% (HIGH), safety=WARNING


Final recommendation: DPP-4
Safety status:        WARNING

assertion passed: Metformin is not the final recommendation


## 5. Batch prediction with an invalid row

Invalid rows are returned as sentinel `PredictionResult(accepted=False, ...)` — the batch does not raise.

In [7]:
good_rows = [generate_patient(rng) for _ in range(3)]
bad_row = {'age': 200, 'bmi': 34}     # malformed

results = engine.predict_batch([*good_rows, bad_row])
for i, r in enumerate(results):
    if r.accepted:
        print(f'{i}: ✓ {r.recommended} (conf={r.confidence_pct}%)')
    else:
        print(f'{i}: ✗ rejected -> {len(r.validation_errors)} field errors')

C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
2026-04-16 09:08:48.268 | INFO     | inference._internal.explainability:extract_safety:674 - Safety warnings for Metformin


2026-04-16 09:08:48.269 | INFO     | inference._internal.explainability:extract:738 - Extracted payload: final=Metformin, orig=Metformin, confidence=100% (HIGH), safety=WARNING


C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


2026-04-16 09:08:48.323 | INFO     | inference._internal.explainability:extract_safety:676 - Safety check CLEAR for Metformin


2026-04-16 09:08:48.324 | INFO     | inference._internal.explainability:extract:738 - Extracted payload: final=Metformin, orig=Metformin, confidence=100% (HIGH), safety=CLEAR


C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


2026-04-16 09:08:48.378 | INFO     | inference._internal.explainability:extract_safety:676 - Safety check CLEAR for DPP-4


2026-04-16 09:08:48.379 | INFO     | inference._internal.explainability:extract:738 - Extracted payload: final=DPP-4, orig=DPP-4, confidence=100% (HIGH), safety=CLEAR


0: ✓ Metformin (conf=100%)
1: ✓ Metformin (conf=100%)
2: ✓ DPP-4 (conf=100%)
3: ✗ rejected -> 15 field errors


## 6. Validation error surface

`ValidationError.errors()` is structured like Pydantic's own — FastAPI-compatible for 422 responses.

In [8]:
try:
    engine.predict({'age': 200, 'bmi': 34})
except ValidationError as e:
    for err in e.errors()[:3]:
        print(err)

{'type': 'less_than_equal', 'loc': ('age',), 'msg': 'Input should be less than or equal to 110', 'input': 200, 'ctx': {'le': 110}, 'url': 'https://errors.pydantic.dev/2.9/v/less_than_equal'}
{'type': 'missing', 'loc': ('hba1c_baseline',), 'msg': 'Field required', 'input': {'age': 200, 'bmi': 34}, 'url': 'https://errors.pydantic.dev/2.9/v/missing'}
{'type': 'missing', 'loc': ('egfr',), 'msg': 'Field required', 'input': {'age': 200, 'bmi': 34}, 'url': 'https://errors.pydantic.dev/2.9/v/missing'}


---

Done. Notebook 13 demonstrates the continuous-learning surface on a synthetic stream.